# Lab 7 - 通常の Agent から Harness Agent へ

この Notebook では、Microsoft Agent Framework の主要 API を Workshop 独自の helper で隠さず、**直接 import・生成**します。最初に Foundry IQ だけを使う通常の Agent を作り、同じ構成へ Toolbox tools、Skills、plan / execute mode、todos、bounded loop を加えて Harness Agent に発展させます。

**使用する kernel:** `Select Kernel > Jupyter Kernel... > Python (Foundry Hosted Agent)`。Codespaces またはローカルの同じ Dev Container で `00-setup.ipynb` を完了してから開きます。コンテナーの再起動後は kernel 内の session が復元されないため、接続先の確認からやり直してください。

| 段階 | 直接記述するもの | 観察すること |
|---|---|---|
| 1 | `AzureCliCredential` → `FoundryChatClient` → `MCPStreamableHTTPTool` → `Agent(...)` | client / instructions / tools / run の最小構成 |
| 2 | `FoundryToolbox` + providers → `create_harness_agent(...)` | task decomposition、Skill の遅延読み込み、Tool Search、複数 tool、todos の完了 |

> **データ境界と料金:** コードは Notebook で動きますが、モデル、Foundry IQ、Toolbox、Code Interpreter、Web Search は Azure 上で実行されます。合成データだけを使い、secret・個人情報・顧客情報を入力しないでください。同じ実行 cell を結果待ちの間に再実行しないでください。

## 1. Lab 1〜4 の接続先を読み込む

`00-setup.ipynb` が生成した `.workshop/context.json` から project、model、Azure AI Search の endpoint を取得し、以降のコンストラクターへ渡す Python 変数にします。Foundry IQ と Toolbox の名前は Lab 3 / 4 で作成した固定名です。API key や client secret は読みません。

In [ ]:
import sys
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "scripts" / "configure_workshop.py"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Repository root が見つかりません。clone した教材内の Notebook を開いてください。"
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.lib.workshop_context import load_context, validate_workshop_context  # noqa: E402
from scripts.lib.workshop_runtime import HOSTED_ENVIRONMENT, require_current_runtime  # noqa: E402

require_current_runtime(HOSTED_ENVIRONMENT)
context_path = REPO_ROOT / ".workshop" / "context.json"
if not context_path.is_file():
    raise FileNotFoundError("00-setup.ipynb を完了し、.workshop/context.json を作成してください。")

context = validate_workshop_context(load_context(context_path))
outputs = context["resource_outputs"]

project_endpoint = outputs["foundry_project_endpoint"]["value"]
model_deployment = outputs["primary_model_deployment_name"]["value"]
search_service_endpoint = outputs["search_service_endpoint"]["value"]
knowledge_base_name = "contoso-travel-knowledge-lab"
toolbox_name = "contoso-travel-toolbox"

print(f"Project: {outputs['foundry_project_name']['value']}")
print(f"Model: {model_deployment}")
print(f"Knowledge base: {knowledge_base_name}")
print(f"Toolbox: {toolbox_name}")

## 2. Foundry IQ を使う通常の Agent を直接作る

次の2セルでは、Agent Framework と Azure Identity から必要なクラスを直接 import し、以下の順番で組み立てます。

```text
AzureCliCredential
  ├─ FoundryChatClient ─┐
  └─ Foundry IQ MCP tool ├─ Agent(...) ─ run(...)
instructions ────────────┘
```

**Middleware** は、処理の前後に共通処理を差し込む仕組みです。Agent Framework には対象ごとの種類があり、この Notebook で使う `ChatMiddleware` は `FoundryChatClient` のモデル呼び出しを包みます。tool / function の実行を包む場合は、別の `FunctionMiddleware` を使います。`process()` 内で `call_next()` を呼ぶと、次の処理へ進み、最終的にモデルへリクエストが送信されます。

この Notebook の `request_pacer` は、モデル呼び出しの開始を最低20秒ずつ離し、40K TPM の deployment で `429 Too Many Requests` が起きる可能性を下げます。通常 Agent と Harness Agent は同じインスタンスを共有します。Foundry IQ や Toolbox の呼び出しは制御せず、Agent Framework の必須機能でもありません。

In [ ]:
import asyncio
import threading
import time
from collections.abc import Awaitable, Callable
from typing import Any

from agent_framework import (
    Agent,
    ChatContext,
    ChatMiddleware,
    MCPStreamableHTTPTool,
)
from agent_framework.foundry import FoundryChatClient, FoundryToolbox
from azure.identity import AzureCliCredential


class ChatRequestPacer(ChatMiddleware):
    def __init__(self, interval_seconds: float = 20.0) -> None:
        self._interval_seconds = interval_seconds
        self._reservation_lock = threading.Lock()
        self._next_start_at = 0.0

    async def process(
        self,
        context: ChatContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        del context
        with self._reservation_lock:
            now = time.monotonic()
            start_at = max(now, self._next_start_at)
            self._next_start_at = start_at + self._interval_seconds
        if start_at > now:
            await asyncio.sleep(start_at - now)
        await call_next()


request_pacer = ChatRequestPacer()

In [ ]:
credential = AzureCliCredential()


def search_headers(tool_arguments: dict[str, Any]) -> dict[str, str]:
    del tool_arguments
    access_token = credential.get_token("https://search.azure.com/.default")
    return {"Authorization": f"Bearer {access_token.token}"}


foundry_iq_url = (
    f"{search_service_endpoint.rstrip('/')}/knowledgebases/{knowledge_base_name}"
    "/mcp?api-version=2026-08-01-preview"
)
standard_chat_client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=model_deployment,
    credential=credential,
    middleware=[request_pacer],
)
standard_iq_tool = MCPStreamableHTTPTool(
    name="contoso-travel-knowledge",
    url=foundry_iq_url,
    header_provider=search_headers,
    allowed_tools=["knowledge_base_retrieve"],
    load_prompts=False,
    approval_mode="never_require",
    request_timeout=120,
)

print(type(standard_chat_client).__name__)
print(f"Foundry IQ MCP: {foundry_iq_url}")

In [ ]:
standard_instructions = """
あなたは Contoso 社内向けの出張・経費アシスタントです。
社内規程に関する質問は Foundry IQ を使って調べ、回答に根拠を付けてください。
確認できない値を推測せず、検索結果にない場合は情報が見つからないと伝えてください。
この Agent はまだ費用計算や承認シミュレーションの tool を持っていません。
""".strip()
standard_agent = Agent(
    client=standard_chat_client,
    name="travel_policy_agent",
    description="Answers Contoso travel-policy questions with Foundry IQ.",
    instructions=standard_instructions,
    tools=[standard_iq_tool],
)

standard_question = (
    "片道12時間の国際線を出発2日前にビジネスクラスで予約したいです。"
    "直前予約の添付物、承認者と順序、申請機能名、標準最大営業日数を、"
    "根拠付きでまとめてください。"
)

async with standard_agent:
    standard_response = await standard_agent.run(standard_question)

print(standard_response.text)

回答で2つの規程の citation を確認してください。この時点で `Agent(...)` の `tools` に渡したのは Foundry IQ だけです。費用計算や現在情報を尋ねても、持っていない tool は実行できません。

| 直接生成した部品 | 役割 |
|---|---|
| `AzureCliCredential()` | コンテナーで `az login` した利用者として認証する |
| `FoundryChatClient(...)` | project の model deployment を呼ぶ |
| `MCPStreamableHTTPTool(...)` | Lab 3 の knowledge base を MCP tool にする |
| `Agent(...)` | client + instructions + tools を実行可能な Agent にまとめる |
| `agent.run(...)` | 1回の依頼を実行する |

## 3. 同じ構成に Harness の実行支援を追加する

通常 Agent と同じ `FoundryChatClient` と Foundry IQ tool を新しく作り、Lab 4 の `FoundryToolbox` と次の provider を明示的に加えます。

- `AgentModeProvider`: plan / execute mode
- `TodoProvider`: session に保存される todos
- `FileSystemAgentFileStore`: `.workshop/harness-memory` に限定した file memory
- `toolbox.as_skills_provider(...)`: Toolbox Skills の progressive disclosure
- `todos_remaining(...)`: 未完了 todo がある execute mode だけを再実行する bounded loop

`create_harness_agent(...)` はこれらを通常の `Agent` に組み込む Agent Framework の factory です。Harness が自動追加できる Web Search は無効にし、Lab 4 の Toolbox と Tool Search を通して利用します。

In [ ]:
harness_instructions = """
あなたは Contoso 社内向けの出張・経費 Harness Agent です。

複雑な依頼は todo に分解し、各 todo を完了してから最終回答を作成してください。
情報源と機能の役割を混同してはいけません。
plan mode では計画と todos の作成だけを行い、Foundry IQ、Skill、Toolbox tool、
Web Search を実行しないでください。ユーザーが計画を承認して execute mode へ移った後に実行します。

- 社内規程と承認手続きの根拠は Foundry IQ で検索し、引用を付ける。
- 操作手順が必要なら Toolbox の Skill を読み込み、その手順に従う。
- Toolbox の tool が必要なら、Tool Search の tool_search で候補を探し、call_tool で実行する。
- tool_search の query は日本語の説明文にせず、目的に合う英語の tool 名を1つ使う。
  見積もりは createTripEstimate、日当照会は getPerDiem、明示された承認シミュレーションは
  createPreapproval、数値計算は code_interpreter、明示された公開情報検索は web_search。
  検索結果の正式な name と inputSchema を確認して call_tool を呼び、未発見の tool を推測しない。
- 費用・日当・承認シミュレーションは Travel Ops API の結果を使い、値を創作しない。
- 比率や複数結果の比較は Code Interpreter を使う。
- Web Search は、現在の公開情報をユーザーが明示的に求めた場合だけ使い、取得時点と出典を示す。

予約や実際の承認は行いません。事前承認シミュレーションはユーザーが明示的に依頼した
場合だけ実行し、実際の承認とは区別してください。必要情報が不足している場合は推測せず、
確認事項を示してください。
""".strip()

In [ ]:
from contextlib import AsyncExitStack

from agent_framework import (
    AgentModeProvider,
    FileSystemAgentFileStore,
    InMemoryHistoryProvider,
    TodoProvider,
    create_harness_agent,
    get_agent_mode,
    set_agent_mode,
    todos_remaining,
    todos_remaining_message,
)

harness_chat_client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=model_deployment,
    credential=credential,
    middleware=[request_pacer],
)
harness_iq_tool = MCPStreamableHTTPTool(
    name="contoso-travel-knowledge",
    url=foundry_iq_url,
    header_provider=search_headers,
    allowed_tools=["knowledge_base_retrieve"],
    load_prompts=False,
    approval_mode="never_require",
    request_timeout=120,
)
toolbox_url = (
    f"{project_endpoint.rstrip('/')}/toolboxes/{toolbox_name}"
    "/mcp?api-version=v1"
)
toolbox = FoundryToolbox(
    credential,
    name=toolbox_name,
    url=toolbox_url,
    load_tools=True,
    approval_mode="never_require",
)
skills_provider = toolbox.as_skills_provider(
    disable_load_skill_approval=True,
    disable_read_skill_resource_approval=True,
)
todo_provider = TodoProvider()
mode_provider = AgentModeProvider(default_mode="plan")
memory_store = FileSystemAgentFileStore(str(REPO_ROOT / ".workshop" / "harness-memory"))

In [ ]:
harness_agent = create_harness_agent(
    client=harness_chat_client,
    name="travel_harness_agent",
    description=(
        "Plans a travel request, retrieves Contoso policy, loads Toolbox Skills, "
        "and uses Tool Search to select the required travel tools."
    ),
    agent_instructions=harness_instructions,
    tools=[harness_iq_tool, toolbox],
    history_provider=InMemoryHistoryProvider(),
    max_context_window_tokens=128_000,
    max_output_tokens=16_384,
    todo_provider=todo_provider,
    mode_provider=mode_provider,
    file_memory_store=memory_store,
    skills_provider=skills_provider,
    disable_web_search=True,
    disable_tool_auto_approval=True,
    loop_should_continue=todos_remaining(looping_modes=["execute"]),
    loop_next_message=todos_remaining_message,
    loop_max_iterations=6,
)

resource_stack = AsyncExitStack()
await resource_stack.__aenter__()
await resource_stack.enter_async_context(harness_agent)
harness_session = harness_agent.create_session()
print(type(harness_agent).__name__)
print(f"Mode: {get_agent_mode(harness_session)}")

### 3-1. 回答と tool call を観察する helper

`AgentResponse.messages` には最終文章だけでなく、モデルが要求した function/MCP call と結果も含まれます。次の helper は content の型と tool 名を一覧にし、回答を書き換えずに観察します。

In [ ]:
from typing import Any


def observed_actions(response: Any) -> list[str]:
    actions = []
    for message in response.messages:
        for content in message.contents:
            name = getattr(content, "name", None)
            if name:
                actions.append(str(name))
    return actions


def current_todos(session: Any) -> list[dict[str, Any]]:
    state = session.state.get(todo_provider.source_id, {})
    return list(state.get("items", [])) if isinstance(state, dict) else []

## 4. plan mode で複雑な依頼を todos に分解する

規程、API、計算、現在情報、承認シミュレーションを 1 回の依頼に含めます。最初は plan mode なので、Harness Agent は実行計画と todos を作り、承認を待ちます。

In [ ]:
from datetime import date, timedelta

start_date = date.today() + timedelta(days=30)
end_date = start_date + timedelta(days=2)
complex_request = f"""
東京からニューヨークへ {start_date.isoformat()} から {end_date.isoformat()} まで、
1名、business class で顧客ワークショップに行く想定です。予算は500,000円です。
社内規程と承認手続きを根拠付きで確認し、Travel Ops API の費用見積もり、
予算との差額と消化率、現在公開されているニューヨーク渡航上の注意情報を1件、
事前承認シミュレーションをまとめてください。現在情報には取得時点と出典を付け、
実際の予約や承認は行わないでください。
""".strip()
print(complex_request)

In [ ]:
plan_response = await harness_agent.run(complex_request, session=harness_session)
print(plan_response.text)
print("\nObserved actions:", observed_actions(plan_response))
print("\nTodos:")
for item in current_todos(harness_session):
    state = "done" if item.get("is_complete") else "open"
    print(f"- [{state}] {item.get('title') or item.get('description')}")

assert get_agent_mode(harness_session) == "plan"
assert current_todos(harness_session), (
    "todo が作成されませんでした。plan の出力を確認してください。"
)

## 5. 計画を承認し、同じ session を execute mode にする

計画に、実際の予約・承認や不要なデータ送信がないことを確認してから次を実行します。`set_agent_mode` は session 内の mode だけを変更します。plan mode で作った todos と会話は同じ session に残ります。

execute mode では未完了 todo が残る場合だけ loop します。`loop_max_iterations=6` が安全上限です。

In [ ]:
set_agent_mode(
    harness_session,
    "execute",
    source_id=mode_provider.source_id,
    available_modes=mode_provider.available_modes,
)
execute_response = await harness_agent.run(
    "この計画を承認します。todos を順に実行し、完了した項目を更新してください。",
    session=harness_session,
)
print(execute_response.text)
execute_actions = observed_actions(execute_response)
print("\nObserved actions:")
for action in execute_actions:
    print(f"- {action}")

todos_after_execute = current_todos(harness_session)
print("\nTodos after execute:")
for item in todos_after_execute:
    state = "done" if item.get("is_complete") else "open"
    print(f"- [{state}] {item.get('title') or item.get('description')}")

assert get_agent_mode(harness_session) == "execute"
assert todos_after_execute
assert all(item.get("is_complete") is True for item in todos_after_execute), todos_after_execute

## 6. 「なぜ複雑な問いを解けたか」を確認する

Observed actions と Portal の Trace を照合し、次の層を区別してください。

| 層 | 期待する記録 | 役割 |
|---|---|---|
| Harness | `todos_add` / `todos_complete`、mode | 依頼の分解と進捗管理 |
| Skills | `load_skill`、skill resource read | 必要な操作手順だけを遅延読み込み |
| Tool Search | `tool_search` → `call_tool` | 必要な tool schema を動的に発見・実行 |
| Foundry IQ | `knowledge_base_retrieve` | 社内規程と承認手続きの根拠 |
| Toolbox tools | Travel Ops / Code Interpreter / Web Search | API 結果、計算、明示的に求めた現在情報 |

Tool Search は task decomposition ではありません。Harness の mode / todos / session / loop が作業を管理し、Tool Search は各 todo に必要な tool を探します。Web Search の内容は変動するため、固定文ではなく取得時点・出典・失敗時の明示を確認します。

In [ ]:
components = {
    "chat client": type(harness_chat_client).__name__,
    "Foundry IQ": type(harness_iq_tool).__name__,
    "Toolbox": type(toolbox).__name__,
    "mode provider": type(mode_provider).__name__,
    "todo provider": type(todo_provider).__name__,
    "memory store": type(memory_store).__name__,
    "Harness Agent": type(harness_agent).__name__,
}
for role, component_type in components.items():
    print(f"{role}: {component_type}")

print("Lab 8 はこの Harness を引き継がず、通常 Agent の sequential workflow と比較します。")

## 7. Notebook の接続を閉じる

Toolbox と Foundry IQ の MCP session、HTTP client を明示的に閉じます。Azure resource は削除しません。resource の cleanup は Lab 9 で行います。

In [ ]:
await resource_stack.aclose()
credential.close()
print("Notebook resources closed.")

## 次の Lab

[Lab 8](../labs/08-hosted-multi-agent.md) では、Luna の token 消費を抑えるため Harness を引き継がず、intake / policy / reviewer の通常 Agent を sequential workflow として Hosted Agent に deploy します。Lab 8 はこの Notebook の session や出力には依存しないため、経験者は Lab 7 を飛ばしても実行できます。